# Inverted Index Creation

### Import liberies

In [2]:
import os
import re
from collections import defaultdict
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import wordnet
from nltk import pos_tag


### Download the NLTK for Lemmetization

In [3]:

# Download resources (only needed once)
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('omw-1.4')
# nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')

nltk.data.path.append(r'C:\nltk_data')  # ✅ Explicitly add correct path

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     C:\Users\hp\AppData\Roaming\nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


### Limmatization

In [5]:

# Initialize lemmatizer
lemmatizer = WordNetLemmatizer()

def tokenize(text):
    """
    Convert text to lowercase and extract alphanumeric tokens.
    """
    text = text.lower()
    tokens = re.findall(r'\w+', text)
    return tokens


### Tokenize

In [ ]:

def is_valid_token(token):
    """
    Accept only tokens that:
    - Are made of ASCII characters
    - Do NOT contain any digits
    """
    return token.isascii() and not any(char.isdigit() for char in token)

def get_wordnet_pos(treebank_tag):
    """
    Map NLTK POS tags to WordNet POS tags.
    """
    if treebank_tag.startswith('J'):
        return wordnet.ADJ
    elif treebank_tag.startswith('V'):
        return wordnet.VERB
    elif treebank_tag.startswith('N'):
        return wordnet.NOUN
    elif treebank_tag.startswith('R'):
        return wordnet.ADV
    else:
        return wordnet.NOUN  # Default fallback

def lemmatize_token(token, pos):
    """
    Lemmatize token using its POS tag.
    """
    return lemmatizer.lemmatize(token, pos)

def get_doc_id_from_filename(filename):
    """
    Extract numeric ID from filename like '123.txt'.
    """
    name_part, ext = os.path.splitext(filename)
    return int(name_part)


### Build Inverted Index

In [ ]:

def build_inverted_index(folder_path):
    """
    Build inverted index from all .txt files using lemmatized tokens.
    """
    inverted_index = defaultdict(set)
    files = sorted(f for f in os.listdir(folder_path) if f.endswith(".txt"))

    for filename in files:
        file_path = os.path.join(folder_path, filename)
        doc_id = get_doc_id_from_filename(filename)

        try:
            with open(file_path, "r", encoding="utf-8") as f:
                text = f.read()
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
            continue

        tokens = tokenize(text)
        filtered_tokens = [token for token in tokens if is_valid_token(token)]

        # POS tagging for valid tokens
        tagged_tokens = pos_tag(filtered_tokens)

        for token, tag in tagged_tokens:
            lemma = lemmatize_token(token, get_wordnet_pos(tag))
            inverted_index[lemma].add(doc_id)

    return inverted_index

def store_inverted_index(inverted_index, output_filename):
    """
    Save the inverted index to a file in readable format.
    """
    with open(output_filename, "w", encoding="utf-8") as f:
        f.write("INVERTED INDEX\n")
        f.write("==============\n")
        f.write("Format: term: doc1, doc2, ...\n\n")
        
        for term in sorted(inverted_index.keys()):
            doc_ids = sorted(list(inverted_index[term]))
            doc_ids_str = ", ".join(str(did) for did in doc_ids)
            f.write(f"{term}: {doc_ids_str}\n")

    print(f"Inverted index stored in '{output_filename}'.")


### Main Program

In [ ]:

if __name__ == "__main__":
    folder_path = r'C:\Users\Public\Dev\Ph.D\2nd Semester\CS-675-IRS\Assignments\First\TXT 1-250'

    if not os.path.exists(folder_path):
        print("The folder path does not exist. Please check the path.")
    else:
        inverted_index = build_inverted_index(folder_path)
        output_file = "inverted_index_lemm.txt"
        store_inverted_index(inverted_index, output_file)